# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MirAziz0/flyrank1/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane 2: Refresh / Content Opportunity Scoring.**

FlyRank's clients have far more published pages than review capacity, and the starter dataset already shows this is a real, sizeable problem rather than a theoretical one: 54.2% of the 30,000 starter pages carry a `trend_direction == "down"` label, and 43.8% of eligible pages (impressions > 0, age >= 90 days) are both declining *and* still receiving meaningful demand (`impressions_90d >= 100`). That is thousands of candidate pages competing for a reviewer's limited attention — exactly the "rank the queue" problem this lane is built for. I'm picking it over Lane 1 (signal analysis) and Lane 3 (clustering) because it produces a directly actionable output (a ranked list with reasons), and over Lane 4 (CTR-only) because the starter pipeline already proves that combining decline signal with visibility, freshness, and position beats a rules-only baseline on this exact data (see Section 3). I'm not picking freestyle because the two core lanes (search signal + content opportunity) fit the starter data's strengths better than a from-scratch AI-referral or growth-prediction direction would this early.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

LANE = "Lane 2: Refresh / Content Opportunity Scoring"
print("Provisional lane:", LANE)


Provisional lane: Lane 2: Refresh / Content Opportunity Scoring


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Research question:** Given a client's existing content inventory, which pages should a content reviewer look at first this week for refresh, expansion, or protection?

- **Unit of analysis:** one content item (`content_id`) per client, summarized over a 90-day observation window — not a query, not a single day.
- **The decision it improves:** which handful of pages, out of a much larger inventory, a human content reviewer spends their limited time on first. Today that choice is made ad hoc or from a hand-tuned rule set; this project turns it into a ranked, evidence-backed queue.
- **Who acts on it:** an SEO/content strategist (internal FlyRank team or client-side) who has capacity to manually review only a small slice of pages per week — the ranking is a triage aid, not an auto-publish trigger.
- **The action:** open the top-ranked pages, read the reason codes (e.g. `declining_with_demand`, `stale_visible_page`, `low_ctr_visible_page`), and decide to refresh, expand, protect, or simply keep monitoring each one.
- **Cost of a wrong call:** two different costs, and both are cheap compared to a bad automated action, because a human still reviews before anything changes. A **false positive** (a page ranked high that didn't need work) costs a reviewer's time — maybe 15-30 minutes reading a page that turns out fine. A **false negative** (a genuinely declining, high-demand page that never surfaces near the top) is more expensive: it keeps losing visibility unnoticed until someone stumbles on it later, which is why the project also reports recall/precision@K, not just precision.
- **Why data/ML helps at all:** with tens of thousands of pages per client and a handful of reviewer-hours per week, nobody can eyeball every page. A transparent baseline plus a model can compress "which 50 pages out of 30,000" into a short, explainable list — something a spreadsheet of raw metrics cannot do at this scale.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
declining_with_demand = (eligible["trend_direction"] == "down") & (eligible["impressions_90d"] >= 100)
candidate_count = int(declining_with_demand.sum())

reviewer_capacity_per_week = 50  # a policy assumption, not a fact in the data
print(f"Eligible pages in the starter slice: {len(eligible):,}")
print(f"Pages flagged declining_with_demand: {candidate_count:,}")
print(f"At {reviewer_capacity_per_week} pages/week reviewer capacity, "
      f"this is a {candidate_count / reviewer_capacity_per_week:.0f}-week backlog if ranked flatly.")


Eligible pages in the starter slice: 30,000
Pages flagged declining_with_demand: 13,152
At 50 pages/week reviewer capacity, this is a 263-week backlog if ranked flatly.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Numbers from `data/raw/content_refresh_anonymized.csv` (30,000 rows) and from the starter pipeline's own committed results (`outputs/model_report.md`):

1. **54.2%** of the 30,000 starter pages carry `trend_direction == "down"`, and **43.8%** of eligible pages (impressions > 0, age >= 90 days) are `declining_with_demand` (down *and* `impressions_90d >= 100`) — the problem this lane targets is common, not rare, in this data.
2. Median `impressions_90d` among eligible pages is **731** and median `content_age_days` is **236** — these are established, visible pages worth reviewing, not thin or brand-new content where a "decline" signal would be noise.
3. The starter pipeline already shows a learned ranking clearly beats a fixed rule on this data: **precision@50 goes from 0.240 (baseline rules) to 0.740 (random forest)** — i.e. about 12 of the baseline's top 50 picks were right versus about 37 of the model's top 50, under client-holdout validation (whole clients held out of training). That gap is the evidence this lane is worth building further, not just running once.

By contrast, only **0.1%** of eligible pages hit the stricter `stale_visible_page` rule (`days_since_last_update >= 180` and `impressions_90d >= 500`) — a reminder that not every baseline rule fires often, and thresholds need checking against real counts before I lean on them (Section 8 of the lane guide).

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("rows:", len(df))
print("pct trend_direction == down:", round((df["trend_direction"] == "down").mean(), 3))

eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
declining_with_demand = (eligible["trend_direction"] == "down") & (eligible["impressions_90d"] >= 100)
stale_visible = (eligible["days_since_last_update"] >= 180) & (eligible["impressions_90d"] >= 500)

print("eligible rows (impressions>0 & age>=90):", len(eligible))
print("pct declining_with_demand:", round(declining_with_demand.mean(), 3))
print("pct stale_visible_page:", round(stale_visible.mean(), 3))
print("median impressions_90d:", eligible["impressions_90d"].median())
print("median content_age_days:", eligible["content_age_days"].median())

# Verified from outputs/model_report.md (committed pipeline output, not re-trained here)
precision_at_50 = {"baseline_rules": 0.240, "random_forest": 0.740}
print("precision@50, baseline vs model:", precision_at_50)


rows: 30000
pct trend_direction == down: 0.542
eligible rows (impressions>0 & age>=90): 30000
pct declining_with_demand: 0.438
pct stale_visible_page: 0.001
median impressions_90d: 731.0
median content_age_days: 236.0
precision@50, baseline vs model: {'baseline_rules': 0.24, 'random_forest': 0.74}


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What this project can say:**
- Which observable signals (impressions, clicks, position, freshness, engagement, trend) are **associated with** a page later being flagged as declining or under-capturing clicks — an observed, directional relationship, not a proven cause.
- A **ranked, decision-support** list: "these pages are the most promising to review first, given limited reviewer time," backed by reason codes a human can inspect and override.
- Whether a learned ranking **out-performs a transparent rule-based baseline** on held-out clients, measured with precision@K and average precision — a real, testable comparison.
- Honest caveats about scope: results come from a 30,000-row anonymized starter slice (or, later, the pseudonymized warehouse release), using whatever labels and windows I define myself and write down in a data contract.

**What this project will never claim:**
- That refreshing a page **caused** a recovery — that requires an experiment or causal design this data alone cannot provide.
- Any claim about **Google's ranking algorithm**, or that the model "predicts Google."
- Any claim about **AI citations or AI search rankings** — the data only measures sessions where someone clicked through from an AI tool, and even that signal is sparse.
- That a "declining" label is a guarantee anything is actually broken — it is a proxy calculated from the current window, and I'll flag it as such rather than as ground truth.
- Anything from a rebuilt product flag (`health_score`, `priority_score`, `action_type`) treated as a feature or as proof of discovery — those are product decisions, not measurements, and are deliberately excluded from the shipped data.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Sanity check: confirm no raw client-identifying or text fields are present in the shipped columns.
unsafe_terms = ["client_name", "url", "domain", "keyword", "query", "title"]
flagged_columns = [c for c in df.columns if any(term in c.lower() for term in unsafe_terms)]
print("Columns matching unsafe-field terms:", flagged_columns)
print("Total columns:", len(df.columns))


Columns matching unsafe-field terms: []
Total columns: 44


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.